In [1]:
import torch
import torch.nn as nn

In [2]:
# Q1. Simple Transformer in PyTorch

d_model = 512
nhead = 8
num_encoder_layers = 6
num_decoder_layers = 6

model = nn.Transformer(
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    batch_first=True
)

print(model)

# Test 
batch_size = 2
src_seq_len = 10
tgt_seq_len = 8

src = torch.rand(batch_size, src_seq_len, d_model)
tgt = torch.rand(batch_size, tgt_seq_len, d_model)

output = model(src, tgt)

print("Input source shape:", src.shape)
print("Input target shape:", tgt.shape)
print("Output shape:", output.shape)

Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, o

In [3]:
# Q2. Embedding Sequences

class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * (self.d_model ** 0.5)


# Test
vocab_size = 10000
d_model = 512

input_embeddings = InputEmbeddings(vocab_size, d_model)

sample_tokens = torch.tensor([[1, 5, 23, 42, 10]])

embedded_tokens = input_embeddings(sample_tokens)

print("Input token IDs shape:", sample_tokens.shape)
print("Embedded output shape:", embedded_tokens.shape)
print("Embedding layer:")
print(input_embeddings)

Input token IDs shape: torch.Size([1, 5])
Embedded output shape: torch.Size([1, 5, 512])
Embedding layer:
InputEmbeddings(
  (embedding): Embedding(10000, 512)
)


In [4]:
# Q3. Positional Encoding

class PositionalEncoding(nn.Module):
    def __init__(self, d_model=512, max_len=5000):
        super().__init__()

        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model)
        )

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]


# Test
batch_size = 2
seq_len = 10
d_model = 512

sample_embeddings = torch.zeros(batch_size, seq_len, d_model)

positional_encoding = PositionalEncoding(d_model=d_model)

output = positional_encoding(sample_embeddings)

print("Input embedding shape:", sample_embeddings.shape)
print("Output shape after positional encoding:", output.shape)
print("Is positional encoding trainable?:", positional_encoding.pe.requires_grad)
print("First position, first 10 values:")
print(output[0, 0, :10])

Input embedding shape: torch.Size([2, 10, 512])
Output shape after positional encoding: torch.Size([2, 10, 512])
Is positional encoding trainable?: False
First position, first 10 values:
tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])


In [5]:
# Q4. Multi-Head Attention

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        Q = self.w_q(query)
        K = self.w_k(key)
        V = self.w_v(value)

        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attention_weights = torch.softmax(scores, dim=-1)

        attention_output = torch.matmul(attention_weights, V)

        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, -1, self.d_model)

        output = self.out_proj(attention_output)

        return output


# Test
batch_size = 2
seq_len = 10
d_model = 512
num_heads = 8

x = torch.randn(batch_size, seq_len, d_model)

multi_head_attention = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

output = multi_head_attention(x, x, x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Output shape matches input shape:", output.shape == x.shape)
print(multi_head_attention)

Input shape: torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
Output shape matches input shape: True
MultiHeadAttention(
  (w_q): Linear(in_features=512, out_features=512, bias=True)
  (w_k): Linear(in_features=512, out_features=512, bias=True)
  (w_v): Linear(in_features=512, out_features=512, bias=True)
  (out_proj): Linear(in_features=512, out_features=512, bias=True)
)


In [6]:
# Q5. Transformer Encoder-Only

class EncoderLayer(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()

        self.self_attention = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        attention_output = self.self_attention(x, x, x)
        x = self.norm1(x + self.dropout1(attention_output))

        feed_forward_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(feed_forward_output))

        return x


class TransformerEncoderBody(nn.Module):
    def __init__(self, d_model=512, num_heads=8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer(d_model=d_model, num_heads=num_heads, d_ff=d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)

        return x


class ClassificationHead(nn.Module):
    def __init__(self, d_model=512, num_classes=3):
        super().__init__()

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        pooled_output = x.mean(dim=1)
        logits = self.classifier(pooled_output)
        probabilities = torch.softmax(logits, dim=-1)

        return probabilities


class EncoderOnlyTransformerClassifier(nn.Module):
    def __init__(self, d_model=512, num_heads=8, num_layers=6, num_classes=3):
        super().__init__()

        self.encoder = TransformerEncoderBody(
            d_model=d_model,
            num_heads=num_heads,
            num_layers=num_layers
        )

        self.classification_head = ClassificationHead(
            d_model=d_model,
            num_classes=num_classes
        )

    def forward(self, x):
        encoder_output = self.encoder(x)
        probabilities = self.classification_head(encoder_output)

        return probabilities


# Test

batch_size = 2
seq_len = 10
d_model = 512
num_heads = 8
num_layers = 6
num_classes = 3

x = torch.randn(batch_size, seq_len, d_model)

encoder_classifier = EncoderOnlyTransformerClassifier(
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    num_classes=num_classes
)

output = encoder_classifier(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Class probabilities:")
print(output)
print("Sum of probabilities for each sample:")
print(output.sum(dim=1))
print(encoder_classifier)

Input shape: torch.Size([2, 10, 512])
Output shape: torch.Size([2, 3])
Class probabilities:
tensor([[0.6130, 0.2157, 0.1713],
        [0.4129, 0.2596, 0.3275]], grad_fn=<SoftmaxBackward0>)
Sum of probabilities for each sample:
tensor([1., 1.], grad_fn=<SumBackward1>)
EncoderOnlyTransformerClassifier(
  (encoder): TransformerEncoderBody(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (self_attention): MultiHeadAttention(
          (w_q): Linear(in_features=512, out_features=512, bias=True)
          (w_k): Linear(in_features=512, out_features=512, bias=True)
          (w_v): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
        (feed_forward

In [7]:
# Q6. Transformer Decoder-Only

class CausalMultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        Q = self.w_q(x)
        K = self.w_k(x)
        V = self.w_v(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        causal_mask = torch.tril(torch.ones(seq_len, seq_len)).to(x.device)
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        scores = scores.masked_fill(causal_mask == 0, float("-inf"))

        attention_weights = torch.softmax(scores, dim=-1)
        attention_output = torch.matmul(attention_weights, V)

        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len, self.d_model)

        output = self.out_proj(attention_output)

        return output


class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()

        self.causal_attention = CausalMultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        attention_output = self.causal_attention(x)
        x = self.norm1(x + self.dropout1(attention_output))

        feed_forward_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(feed_forward_output))

        return x


class TransformerDecoderBody(nn.Module):
    def __init__(self, d_model=512, num_heads=8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer(d_model=d_model, num_heads=num_heads, d_ff=d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)

        return x


class LanguageModelHead(nn.Module):
    def __init__(self, d_model=512, vocab_size=10000):
        super().__init__()

        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        logits = self.output_layer(x)
        probabilities = torch.softmax(logits, dim=-1)

        return probabilities


class DecoderOnlyTransformerLM(nn.Module):
    def __init__(self, vocab_size=10000, d_model=512, num_heads=8, num_layers=6):
        super().__init__()

        self.embedding = InputEmbeddings(vocab_size=vocab_size, d_model=d_model)
        self.positional_encoding = PositionalEncoding(d_model=d_model)

        self.decoder = TransformerDecoderBody(
            d_model=d_model,
            num_heads=num_heads,
            num_layers=num_layers
        )

        self.lm_head = LanguageModelHead(
            d_model=d_model,
            vocab_size=vocab_size
        )

    def forward(self, token_ids):
        x = self.embedding(token_ids)
        x = self.positional_encoding(x)
        x = self.decoder(x)
        probabilities = self.lm_head(x)

        return probabilities


# Test

batch_size = 2
seq_len = 10
vocab_size = 10000
d_model = 512
num_heads = 8
num_layers = 6

sample_token_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

decoder_lm = DecoderOnlyTransformerLM(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers
)

output = decoder_lm(sample_token_ids)

print("Input token IDs shape:", sample_token_ids.shape)
print("Output shape:", output.shape)
print("Expected output shape:", (batch_size, seq_len, vocab_size))
print("Output shape is correct:", output.shape == (batch_size, seq_len, vocab_size))

print("Sum of probabilities for first sample, first token:")
print(output[0, 0, :].sum())

print(decoder_lm)

Input token IDs shape: torch.Size([2, 10])
Output shape: torch.Size([2, 10, 10000])
Expected output shape: (2, 10, 10000)
Output shape is correct: True
Sum of probabilities for first sample, first token:
tensor(1.0000, grad_fn=<SumBackward0>)
DecoderOnlyTransformerLM(
  (embedding): InputEmbeddings(
    (embedding): Embedding(10000, 512)
  )
  (positional_encoding): PositionalEncoding()
  (decoder): TransformerDecoderBody(
    (layers): ModuleList(
      (0-5): 6 x DecoderLayer(
        (causal_attention): CausalMultiHeadAttention(
          (w_q): Linear(in_features=512, out_features=512, bias=True)
          (w_k): Linear(in_features=512, out_features=512, bias=True)
          (w_v): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropo

In [ ]:
# Q7. OpenAI GPT-4o API

# Variant A: Official OpenAI SDK
from openai import OpenAI

client = OpenAI(
    api_key="YOUR_OPENAI_API_KEY"
)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": "Give me a short explanation of the rules of American football."
        }
    ],
    temperature=0.7,
    max_tokens=200
)

print(response.choices[0].message.content)


# Variant B: LangChain connection
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",
    api_key="YOUR_OPENAI_API_KEY",
    temperature=0.7,
    max_tokens=200
)

response = llm.invoke(
    "Give me a short explanation of the rules of American football."
)

print(response.content)

In [ ]:
# Q8. Anthropic Claude API

# Variant A: Official Anthropic SDK
import anthropic

client = anthropic.Anthropic(
    api_key="YOUR_ANTHROPIC_API_KEY"
)

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=220,
    temperature=0.7,
    messages=[
        {
            "role": "user",
            "content": "Explain why humans dream during sleep."
        }
    ]
)

print(response.content[0].text)


# Variant B: LangChain connection
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    api_key="YOUR_ANTHROPIC_API_KEY",
    temperature=0.7,
    max_tokens=220
)

response = llm.invoke(
    "Explain why humans dream during sleep."
)

print(response.content)

In [ ]:
# Q9. Google Gemini API

# Variant A: Official Google Generative AI SDK
import google.generativeai as genai

genai.configure(
    api_key="YOUR_GEMINI_API_KEY"
)

model = genai.GenerativeModel("gemini-1.5-flash")

response = model.generate_content(
    "What could future cities look like?"
)

print(response.text)


# Variant B: LangChain connection
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key="YOUR_GEMINI_API_KEY"
)

response = llm.invoke(
    "What could future cities look like?"
)

print(response.content)

In [ ]:
# Q10. xAI Grok API

# Variant A: xAI Grok API using the OpenAI-compatible endpoint
from openai import OpenAI

client = OpenAI(
    api_key="YOUR_XAI_API_KEY",
    base_url="https://api.x.ai/v1"
)

response = client.chat.completions.create(
    model="grok-4.3",
    messages=[
        {
            "role": "user",
            "content": "What are the benefits of reading books?"
        }
    ],
    temperature=0.7,
    max_tokens=200
)

print(response.choices[0].message.content)


# Variant B: LangChain connection
from langchain_xai import ChatXAI

llm = ChatXAI(
    model="grok-4.3",
    xai_api_key="YOUR_XAI_API_KEY",
    temperature=0.7,
    max_tokens=200
)

response = llm.invoke(
    "What are the benefits of reading books?"
)

print(response.content)